# Chapter 5 — The Adjoint in Linear Algebra
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Implement the adjoint method for linear systems. Verify Theorem 5.1 (adjoint = forward SA). Illustrate the LP duality connection and cost comparison.

**Key claims tested:**
- Adjoint gives identical SI as direct solve (Theorem 5.1); cost is r solves vs m solves; LP duality: A^T v = c is dual constraint

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
FULL = False
print('Chapter 5: The Adjoint in Linear Algebra')

In [ ]:
# ── Forward vs Adjoint for linear system Au = b ───────────────────────────────
# Example from §5.1: A = [[3,1],[1,2]], b depends on parameters q1, q2
# Response: J = u1 + u2 = c^T u, c = [1,1]

A = np.array([[3.0, 1.0], [1.0, 2.0]])
c = np.array([1.0, 1.0])

# Forward (FSE): one solve per parameter
b = lambda q: np.array(q)
q_nom = np.array([1.0, 1.0])
u_nom = np.linalg.solve(A, b(q_nom))
J_nom = c @ u_nom

db_dq1 = np.array([1.0, 0.0])  # db/dq1
db_dq2 = np.array([0.0, 1.0])  # db/dq2
du_dq1 = np.linalg.solve(A, db_dq1)  # FSE solve 1
du_dq2 = np.linalg.solve(A, db_dq2)  # FSE solve 2
dJ_dq1_fwd = c @ du_dq1
dJ_dq2_fwd = c @ du_dq2

# Adjoint: ONE solve regardless of number of parameters
v = np.linalg.solve(A.T, c)          # A^T v = c (one solve)
dJ_dq1_adj = v @ db_dq1              # dot product only
dJ_dq2_adj = v @ db_dq2              # dot product only

print('Forward SA (2 solves):')
print(f'  dJ/dq1 = {dJ_dq1_fwd:.6f}  dJ/dq2 = {dJ_dq2_fwd:.6f}')
print('Adjoint SA (1 solve + 2 dot products):')
print(f'  dJ/dq1 = {dJ_dq1_adj:.6f}  dJ/dq2 = {dJ_dq2_adj:.6f}')
err = max(abs(dJ_dq1_fwd-dJ_dq1_adj), abs(dJ_dq2_fwd-dJ_dq2_adj))
assert err < 1e-12, f'FAIL: err={err}'
print(f'PASS: Forward = Adjoint (Theorem 5.1), max diff = {err:.2e}')
print(f'Adjoint vector v = {v}  (encodes weighted contributions to J)')

In [ ]:
# ── Verification: Theorem 5.1, the adjoint identity, and the cost claim ──────
# Self-contained apart from variables defined in the cell above.
# Every quantity is checked against a closed form, not against itself.
import numpy as np

_tol = 1e-12

# (1) Closed form. det(A) = 5, so A^{-1} = (1/5)[[2,-1],[-1,3]] and, with
#     c = b(q_nom) = [1,1], the exact sensitivities are 1/5 and 2/5.
_Ainv = np.array([[2.0, -1.0], [-1.0, 3.0]]) / 5.0
assert np.allclose(_Ainv @ A, np.eye(2), atol=_tol), "closed-form inverse is wrong"
_exact = np.array([1.0/5.0, 2.0/5.0])

# (2) Theorem 5.1: the two routes agree, and both agree with the closed form.
_fwd = np.array([dJ_dq1_fwd, dJ_dq2_fwd])
_adj = np.array([dJ_dq1_adj, dJ_dq2_adj])
assert np.max(np.abs(_fwd - _adj)) < _tol, (
    f"forward and adjoint disagree by {np.max(np.abs(_fwd-_adj)):.3e}")
assert np.max(np.abs(_adj - _exact)) < _tol, (
    f"adjoint route gives {_adj}, closed form gives {_exact}")
assert abs(J_nom - 3.0/5.0) < _tol, f"J_nom = {J_nom}, expected 3/5"

# (3) The adjoint variable satisfies its defining equation A^T v = c.
assert np.max(np.abs(A.T @ v - c)) < _tol, "A^T v = c is not satisfied"

# (4) The cost claim of the chapter: forward needs one solve per parameter,
#     adjoint needs one solve per response, independent of m.
_m, _r = 2, 1                      # 2 parameters (q1, q2), 1 response J
_forward_solves, _adjoint_solves = _m, _r
assert _adjoint_solves < _forward_solves, (
    "with m > r the adjoint must use strictly fewer solves")

print(f"Verification passed: dJ/dq = {_adj} = exact [1/5, 2/5]; "
      f"A^T v = c to {np.max(np.abs(A.T @ v - c)):.1e}; "
      f"{_adjoint_solves} adjoint solve vs {_forward_solves} forward.")


In [ ]:
# ── Cost comparison figure ────────────────────────────────────────────────────
m_vals = np.arange(1, 51)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(m_vals, m_vals, 'c-', lw=2, label='Forward SA: m linear solves')
for r, col, ls in [(1,'steelblue','-'),(3,'coral','--'),(5,'seagreen',':')]:
    ax.axhline(r, color=col, lw=2, ls=ls, label=f'Adjoint: r={r} QOI(s)')
ax.set_xlabel('Number of POIs (m)', fontsize=12)
ax.set_ylabel('Number of linear solves', fontsize=12)
ax.set_title('Adjoint vs Forward SA — Cost Comparison\n(Theorem 5.1: same answer, lower cost)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim(1,50); ax.set_ylim(0,51)
plt.tight_layout()
plt.savefig('ch05_fig_cost.pdf', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 5 saved.')

In [ ]:
try:
    from google.colab import files
    files.download('ch05_fig_cost.pdf')
except ImportError:
    print('Not in Colab — figure saved locally.')